# 07 — Chronological Train / Validation / Test Splitting

**Objective:** Freeze the locked chronological evaluation partitioning:
- **Development / Training:** 2011–2013 (22,191 records)
- **Calibration / Validation:** 2014 (7,158 records)
- **Locked Test Set:** 2015 (7,384 records) — strictly protected from model tuning.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.splitting import temporal_split, verify_leakage_free


## 1. Partition Verification & Leakage Prevention


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev_df, cal_df, test_df = temporal_split(df)

split_manifest = {
    "development": {
        "years": [2011, 2012, 2013],
        "rows": len(dev_df),
        "percentage": round(len(dev_df) / len(df) * 100, 2)
    },
    "calibration": {
        "years": [2014],
        "rows": len(cal_df),
        "percentage": round(len(cal_df) / len(df) * 100, 2)
    },
    "locked_test": {
        "years": [2015],
        "rows": len(test_df),
        "percentage": round(len(test_df) / len(df) * 100, 2)
    },
    "leakage_audit": verify_leakage_free(dev_df, test_df)
}

print(json.dumps(split_manifest, indent=2))

out_split = ROOT / "data" / "processed" / "split_manifest.json"
with open(out_split, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)

print(f"Saved split manifest to {out_split.relative_to(ROOT)}")
print("Train/Validation/Test splitting COMPLETE.")


{
  "development": {
    "years": [
      2011,
      2012,
      2013
    ],
    "rows": 22191,
    "percentage": 60.41
  },
  "calibration": {
    "years": [
      2014
    ],
    "rows": 7158,
    "percentage": 19.49
  },
  "locked_test": {
    "years": [
      2015
    ],
    "rows": 7384,
    "percentage": 20.1
  },
  "leakage_audit": {
    "disjoint_years": true,
    "overlapping_years": [],
    "train_rows": 22191,
    "test_rows": 7384,
    "leakage_detected": false
  }
}
Saved split manifest to data\processed\split_manifest.json
Train/Validation/Test splitting COMPLETE.
